In [96]:
#@title The Monte Carlo TreeNode class
# prompt: make me a recursive tree structure where weach node has pointers to its children and a pointer to its parent. it should keep track of the number of times visited as well as a value we give it
class TreeNode:
  import numpy as np

  def __init__(self, pos, parent):
    self.pos = pos
    self.parent = parent
    self.children_nodes = {}
    self.children_edges = {}
    self.visit_count = 0
    self.value = 0

  def __str__(self, level=0, max_level=3):
    repr = (("  " * level) + f"Node: pos={self.pos}, value={self.value}")
    if level > max_level:
      return repr
    for child_node in self.children_nodes.values():
      repr += ("\n" + child_node.__str__(level + 1))
    return repr

  def __repr__(self) -> str:
    return self.__str__(level=0, max_level=6)

  def __getitem__(self, args):
    Q, childIindex = args
    if childIindex not in self.children_nodes:
      self.children_nodes[childIindex] = TreeNode(childIindex, self)
      self.children_edges[childIindex] = Q[childIindex, self.pos]
    return self.children_nodes[childIindex], self.children_edges[childIindex]

  def update_value(self, target):

    # net flow comes from each child
    self.value = np.sum([self.children_nodes[j].value * self.children_edges[j] for j in self.children_nodes])

    # if this is a target node, we get a unit of flow
    if self.pos == target:
      self.value += 1

  def rollout(self, Q, target, curr_flow, epsilon):

    # select a random child to roll out a sample / update
    childIindex = np.random.choice(np.nonzero(Q[:, self.pos])[0])
    child, edge = self[Q, childIindex]
    new_flow = curr_flow * edge

    # if we reach the epsilon horizon, stop here. Stop anytime = try-catch
    if abs(new_flow) > epsilon:
      try:
        child.rollout(Q, target, new_flow, epsilon)
      except:
        pass

    # update node's value
    self.update_value(target)

In [97]:
#@title The Monte Carlo Tree Search driver
def estimate_entry(row, col, epsilon=1e-6, num_iters=int(1e5), verbose=0):
  root = TreeNode(pos=col, parent=None)
  if verbose:
    from tqdm import trange
    for _ in trange(num_iters):
      root.rollout(Q, row, 1, epsilon)
  else:
    for _ in range(num_iters):
      root.rollout(Q, row, 1, epsilon)
  if verbose > 1:
    print()
    print(root)
  return root.value

# Workspace

---

In [98]:
import numpy as np
n = 5
Q = 0.1 * np.random.random((n,n))
M_inv = np.linalg.pinv(np.eye(n) - Q)

In [99]:
print(Q)
print(M_inv)

[[0.09226709 0.09766584 0.03631684 0.03557492 0.04026808]
 [0.00725105 0.022415   0.01435025 0.0273015  0.02585843]
 [0.05399232 0.00932949 0.00677122 0.06830619 0.06056727]
 [0.09428632 0.01217402 0.06646791 0.02274211 0.03918861]
 [0.07390159 0.02935565 0.0743167  0.011249   0.04969439]]
[[1.11501155 0.11413358 0.04977677 0.04789611 0.05550058]
 [0.01511796 1.02594719 0.01977364 0.03095207 0.03109395]
 [0.07453029 0.02027934 1.02025836 0.07541839 0.07184571]
 [0.11661928 0.02687467 0.07785758 1.03432272 0.0532884 ]
 [0.09438607 0.04247202 0.0851906  0.0228223  1.06381898]]


In [100]:
estimate = np.array([[estimate_entry(i, j, epsilon=1e-6, num_iters=100) for j in range(Q.shape[1])] for i in range(Q.shape[0])])
print("Percent Error In Each Entry:")
print(100 * ((M_inv - estimate) / M_inv).round(5))

Percent Error In Each Entry:
[[0.309 1.605 3.746 2.417 3.172]
 [3.724 0.055 2.307 1.785 1.885]
 [2.137 6.946 0.145 1.367 1.945]
 [1.758 5.545 2.237 0.189 3.341]
 [3.415 2.112 1.97  6.764 0.182]]


In [101]:
estimate_entry(2, 3, epsilon=1e-6, num_iters=1000, verbose=2)

100%|██████████| 1000/1000 [00:00<00:00, 7206.84it/s]


Node: pos=3, value=0.07524406965525758
  Node: pos=1, value=0.01930361185273685
    Node: pos=2, value=1.0158125534646232
      Node: pos=3, value=0.06830618967029466
        Node: pos=4, value=0
        Node: pos=2, value=1.0
        Node: pos=0, value=0
        Node: pos=1, value=0
        Node: pos=3, value=0
      Node: pos=2, value=1.0
        Node: pos=4, value=0
        Node: pos=1, value=0
        Node: pos=2, value=0
      Node: pos=4, value=0.0605672666851059
        Node: pos=2, value=1.0
        Node: pos=0, value=0
        Node: pos=3, value=0
        Node: pos=1, value=0
        Node: pos=4, value=0
      Node: pos=0, value=0.0
        Node: pos=0, value=0
        Node: pos=4, value=0
        Node: pos=1, value=0
        Node: pos=2, value=0
        Node: pos=3, value=0
      Node: pos=1, value=0.0
        Node: pos=0, value=0
        Node: pos=3, value=0
        Node: pos=1, value=0
        Node: pos=2, value=0
        Node: pos=4, value=0
    Node: pos=4, value=0.06274

np.float64(0.07524406965525758)